# 💼 Salary Prediction System — Regression Models Only

**CS1138 Machine Learning | Group 9 | JK Lakshmipat University**  
**Dataset:** Salary Prediction Dataset (1000 records)

### Regression Models Covered:
1. **Linear Regression** — with all key parameters
2. **Ridge Regression** — L2 regularisation, all parameters
3. **Lasso Regression** — L1 regularisation, all parameters
4. **ElasticNet Regression** — combined L1+L2, all parameters
5. **Decision Tree Regressor** — all tree parameters
6. **Random Forest Regressor** — all ensemble parameters
7. **Gradient Boosting Regressor** — all boosting parameters
8. **Support Vector Regressor (SVR)** — all kernel parameters
9. **K-Nearest Neighbours Regressor (KNR)** — all distance parameters
10. **Polynomial Regression** — degree + LinearRegression parameters

### Evaluation Metrics:
- MAE, MSE, RMSE, R², Adjusted R², MAPE

### Extras:
- Full EDA with updated styled graphs
- GridSearchCV hyperparameter tuning (Ridge, Lasso, ElasticNet, GBR)
- Residual analysis plots
- Feature importance plots
- Prediction interface

## STEP 1 — Install & Import Libraries

In [ ]:
# ══════════════════════════════════════════════════════════════════
# !pip install scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (
    StandardScaler, LabelEncoder, PolynomialFeatures
)
from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score, KFold
)
from sklearn.pipeline import Pipeline

# ── All Regression Models ──────────────────────────────────────────
from sklearn.linear_model import (
    LinearRegression,   # OLS regression
    Ridge,              # L2 regularisation
    Lasso,              # L1 regularisation
    ElasticNet,         # Combined L1+L2
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# ── Metrics ───────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

# ── Plot style ────────────────────────────────────────────────────
PALETTE  = ['#2C5F2D','#065A82','#F96167','#028090','#6D2E46',
            '#B85042','#E8A838','#4B4E6D','#A8DADC','#457B9D']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'xtick.labelsize' : 9,
    'ytick.labelsize' : 9,
    'figure.dpi'      : 120,
})

print('✅ All libraries imported successfully!')

## STEP 2 — Load Dataset

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Upload salary_prediction_data.csv in Colab before running
df = pd.read_csv('/content/salary_prediction_data.csv')

print(f'✅ Loaded! Shape: {df.shape}')
print(f'Columns : {list(df.columns)}')
df.head()

## STEP 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3A: Dataset overview
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EDA Plot 1 — Numerical Feature Distributions (KDE + Histogram)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('📊 Numerical Feature Distributions', fontsize=15, fontweight='bold', y=1.02)

for ax, col, color in zip(axes,
                           ['Salary', 'Age', 'Experience'],
                           ['#2C5F2D', '#065A82', '#F96167']):
    sns.histplot(df[col], bins=30, kde=True, color=color,
                 ax=ax, edgecolor='white', alpha=0.82, linewidth=0.6)
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val,   color='crimson',  linestyle='--', lw=1.6,
               label=f'Mean: {mean_val:,.1f}')
    ax.axvline(median_val, color='navy',     linestyle=':',  lw=1.6,
               label=f'Median: {median_val:,.1f}')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col + (' (USD)' if col == 'Salary' else ''))
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_1_numerical_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 1 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EDA Plot 2 — Categorical Feature Distributions
cat_cols = ['Education', 'Job_Title', 'Location', 'Gender']
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('📊 Categorical Feature Distributions', fontsize=15, fontweight='bold', y=1.02)

for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=PALETTE[:len(counts)], edgecolor='white', linewidth=0.7)
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
                str(int(bar.get_height())), ha='center', fontsize=8, fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig('eda_2_categorical.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 2 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EDA Plot 3 — Salary vs Categorical Features (violin + strip)
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('💰 Salary Distribution by Categorical Features',
             fontsize=15, fontweight='bold', y=1.02)

for ax, col in zip(axes, cat_cols):
    order = df.groupby(col)['Salary'].median().sort_values(ascending=False).index
    sns.violinplot(data=df, x=col, y='Salary', order=order,
                   palette=PALETTE[:df[col].nunique()],
                   inner='quartile', ax=ax, alpha=0.8)
    sns.stripplot(data=df, x=col, y='Salary', order=order,
                  color='white', size=1.5, jitter=True, ax=ax, alpha=0.3)
    ax.set_title(f'Salary by {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Salary (USD)' if col == 'Education' else '')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'${x/1000:.0f}K'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig('eda_3_salary_vs_categorical.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 3 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EDA Plot 4 — Correlation heatmap + Pair plot for numeric cols
num_cols = ['Age', 'Experience', 'Salary']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🔗 Correlation Analysis', fontsize=14, fontweight='bold')

# Heatmap
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn',
            ax=axes[0], square=True, linewidths=1, vmin=-1, vmax=1,
            annot_kws={'size': 12, 'weight': 'bold'})
axes[0].set_title('Correlation Matrix (Numerical Features)', fontweight='bold')

# Salary vs Experience scatter coloured by Education
edu_codes = pd.Categorical(df['Education']).codes
sc = axes[1].scatter(df['Experience'], df['Salary'],
                     c=edu_codes, cmap='tab10',
                     alpha=0.65, s=25, edgecolors='none')
cb = plt.colorbar(sc, ax=axes[1])
cb.set_label('Education (encoded)', fontsize=9)
axes[1].set_title('Salary vs Experience (colour = Education)', fontweight='bold')
axes[1].set_xlabel('Years of Experience')
axes[1].set_ylabel('Salary (USD)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(
    lambda x, _: f'${x/1000:.0f}K'))

# Regression line overlay
m, b = np.polyfit(df['Experience'], df['Salary'], 1)
xs = np.linspace(df['Experience'].min(), df['Experience'].max(), 100)
axes[1].plot(xs, m*xs + b, color='crimson', lw=2, linestyle='--', label='OLS trend')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('eda_4_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 4 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EDA Plot 5 — Salary boxplot by Education × Gender + Mean salary by Job
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('💰 Salary Deep-Dive', fontsize=14, fontweight='bold')

sns.boxplot(data=df, x='Education', y='Salary', hue='Gender',
            palette=['#065A82', '#F96167'], ax=axes[0],
            width=0.5, flierprops=dict(marker='o', markersize=2, alpha=0.4))
axes[0].set_title('Salary by Education & Gender', fontweight='bold')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(
    lambda x, _: f'${x/1000:.0f}K'))
axes[0].legend(title='Gender', fontsize=9)

avg_job = df.groupby('Job_Title')['Salary'].mean().sort_values()
colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(avg_job))]
bars = axes[1].barh(avg_job.index, avg_job.values, color=colors_bar, edgecolor='white')
axes[1].set_title('Mean Salary by Job Title', fontweight='bold')
axes[1].set_xlabel('Average Salary (USD)')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(
    lambda x, _: f'${x/1000:.0f}K'))
for bar in bars:
    w = bar.get_width()
    axes[1].text(w + 300, bar.get_y() + bar.get_height()/2,
                 f'${w:,.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('eda_5_salary_deepdive.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 5 saved!')

## STEP 4 — Preprocessing & Feature Engineering

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4A: Label-encode categorical columns
le_edu = LabelEncoder()
le_loc = LabelEncoder()
le_job = LabelEncoder()
le_gen = LabelEncoder()

df['Education_enc'] = le_edu.fit_transform(df['Education'])
df['Location_enc']  = le_loc.fit_transform(df['Location'])
df['Job_Title_enc'] = le_job.fit_transform(df['Job_Title'])
df['Gender_enc']    = le_gen.fit_transform(df['Gender'])

print('Label Encoding mappings:')
for le, col in [(le_edu,'Education'),(le_loc,'Location'),
                (le_job,'Job_Title'),(le_gen,'Gender')]:
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# 4B: Feature matrix & regression target (exact salary)
FEATURE_COLS = ['Education_enc', 'Experience', 'Location_enc',
                'Job_Title_enc', 'Age', 'Gender_enc']
FEAT_LABELS  = ['Education', 'Experience', 'Location',
                'Job Title', 'Age', 'Gender']

X = df[FEATURE_COLS].values
y = df['Salary'].values
print(f'\n✅ Feature matrix X: {X.shape}')
print(f'✅ Target y (Salary) — min: ${y.min():,.0f}  max: ${y.max():,.0f}  mean: ${y.mean():,.0f}')

# 4C: Standardise features
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4D: 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f'\nTrain: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print('✅ Preprocessing complete!')

## STEP 5 — Train All Regression Models (Full Parameters)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 1: LINEAR REGRESSION — Ordinary Least Squares
# Parameters:
#   fit_intercept : bool   — whether to fit an intercept term  (default True)
#   copy_X        : bool   — copy X before fitting             (default True)
#   n_jobs        : int    — parallel jobs (-1 = all cores)    (default None)
#   positive      : bool   — force non-negative coefficients   (default False)

lr = LinearRegression(
    fit_intercept=True,
    copy_X=True,
    n_jobs=-1,
    positive=False,
)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print(f'✅ Linear Regression        | R²={r2_score(y_test, y_pred_lr):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 2: RIDGE REGRESSION — L2 Regularisation
# Parameters:
#   alpha         : float  — regularisation strength (larger = more)  (default 1.0)
#   fit_intercept : bool   — fit intercept                            (default True)
#   copy_X        : bool   — copy X                                   (default True)
#   max_iter      : int    — max iterations for solver                (default None)
#   tol           : float  — convergence tolerance                    (default 1e-3)
#   solver        : str    — 'auto','svd','cholesky','lsqr','sparse_cg','sag','saga'
#   random_state  : int    — for 'sag','saga' solvers                 (default None)

ridge = Ridge(
    alpha=1.0,
    fit_intercept=True,
    copy_X=True,
    max_iter=None,
    tol=1e-3,
    solver='auto',
    random_state=42,
)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
print(f'✅ Ridge Regression         | R²={r2_score(y_test, y_pred_ridge):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 3: LASSO REGRESSION — L1 Regularisation (feature selection)
# Parameters:
#   alpha         : float  — regularisation strength                 (default 1.0)
#   fit_intercept : bool   — fit intercept                           (default True)
#   precompute    : bool   — precompute Gram matrix                  (default False)
#   copy_X        : bool   — copy X                                  (default True)
#   max_iter      : int    — max iterations                          (default 1000)
#   tol           : float  — convergence tolerance                   (default 1e-4)
#   warm_start    : bool   — reuse previous solution                 (default False)
#   positive      : bool   — force positive coefficients             (default False)
#   selection     : str    — 'cyclic' or 'random' (coord descent)   (default 'cyclic')
#   random_state  : int    — for 'random' selection                  (default None)

lasso = Lasso(
    alpha=0.1,
    fit_intercept=True,
    precompute=False,
    copy_X=True,
    max_iter=1000,
    tol=1e-4,
    warm_start=False,
    positive=False,
    selection='cyclic',
    random_state=42,
)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print(f'✅ Lasso Regression         | R²={r2_score(y_test, y_pred_lasso):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 4: ELASTICNET REGRESSION — Combined L1 + L2
# Parameters:
#   alpha         : float  — overall regularisation strength         (default 1.0)
#   l1_ratio      : float  — mix ratio (0 = Ridge, 1 = Lasso)        (default 0.5)
#   fit_intercept : bool   — fit intercept                           (default True)
#   precompute    : bool   — precompute Gram matrix                  (default False)
#   max_iter      : int    — max iterations                          (default 1000)
#   copy_X        : bool   — copy X                                  (default True)
#   tol           : float  — convergence tolerance                   (default 1e-4)
#   warm_start    : bool   — reuse previous solution                 (default False)
#   positive      : bool   — force positive coefficients             (default False)
#   selection     : str    — 'cyclic' or 'random'                   (default 'cyclic')
#   random_state  : int    — for 'random' selection                  (default None)

enet = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    fit_intercept=True,
    precompute=False,
    max_iter=1000,
    copy_X=True,
    tol=1e-4,
    warm_start=False,
    positive=False,
    selection='cyclic',
    random_state=42,
)
enet.fit(X_train, y_train)
y_pred_enet = enet.predict(X_test)
print(f'✅ ElasticNet Regression    | R²={r2_score(y_test, y_pred_enet):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 5: POLYNOMIAL REGRESSION (degree=2) via Pipeline
# Parameters for PolynomialFeatures:
#   degree            : int   — degree of polynomial features         (default 2)
#   interaction_only  : bool  — only interaction terms, no powers     (default False)
#   include_bias      : bool  — include bias column of 1s             (default True)
#   order             : str   — 'C' (row-major) or 'F' (col-major)   (default 'C')
# Then wrapped with LinearRegression (same params as MODEL 1)

poly_reg = Pipeline([
    ('poly', PolynomialFeatures(
        degree=2,
        interaction_only=False,
        include_bias=True,
        order='C',
    )),
    ('lr', LinearRegression(
        fit_intercept=True,
        copy_X=True,
        n_jobs=-1,
        positive=False,
    )),
])
poly_reg.fit(X_train, y_train)
y_pred_poly = poly_reg.predict(X_test)
print(f'✅ Polynomial Regression    | R²={r2_score(y_test, y_pred_poly):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 6: DECISION TREE REGRESSOR
# Parameters:
#   criterion        : str   — 'squared_error','friedman_mse','absolute_error','poisson'
#   splitter         : str   — 'best' or 'random'                   (default 'best')
#   max_depth        : int   — max tree depth (None = full)          (default None)
#   min_samples_split: int   — min samples to split a node           (default 2)
#   min_samples_leaf : int   — min samples in a leaf                 (default 1)
#   min_weight_fraction_leaf: float — fraction of weights            (default 0.0)
#   max_features     : str   — 'auto','sqrt','log2',int,float,None   (default None)
#   random_state     : int   — random seed                           (default None)
#   max_leaf_nodes   : int   — max leaf nodes (None = unlimited)     (default None)
#   min_impurity_decrease: float — min impurity decrease to split    (default 0.0)
#   ccp_alpha        : float — complexity param for pruning          (default 0.0)

dt_reg = DecisionTreeRegressor(
    criterion='squared_error',
    splitter='best',
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    random_state=42,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
)
dt_reg.fit(X_train, y_train)
y_pred_dt = dt_reg.predict(X_test)
print(f'✅ Decision Tree Regressor  | R²={r2_score(y_test, y_pred_dt):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 7: RANDOM FOREST REGRESSOR
# Parameters:
#   n_estimators     : int   — number of trees                       (default 100)
#   criterion        : str   — 'squared_error','absolute_error','friedman_mse','poisson'
#   max_depth        : int   — max tree depth                        (default None)
#   min_samples_split: int   — min samples to split                  (default 2)
#   min_samples_leaf : int   — min samples in leaf                   (default 1)
#   min_weight_fraction_leaf: float                                  (default 0.0)
#   max_features     : str   — '1.0','sqrt','log2',int,float         (default 1.0)
#   max_leaf_nodes   : int   — max leaf nodes                        (default None)
#   min_impurity_decrease: float                                     (default 0.0)
#   bootstrap        : bool  — bootstrap samples                     (default True)
#   oob_score        : bool  — out-of-bag score                      (default False)
#   n_jobs           : int   — parallel jobs                         (default None)
#   random_state     : int                                           (default None)
#   warm_start       : bool  — reuse trees from prev fit             (default False)
#   ccp_alpha        : float — pruning param                         (default 0.0)
#   max_samples      : int   — samples per tree if bootstrap=True    (default None)

rf_reg = RandomForestRegressor(
    n_estimators=200,
    criterion='squared_error',
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=3,
    min_weight_fraction_leaf=0.0,
    max_features=1.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=42,
    warm_start=False,
    ccp_alpha=0.0,
    max_samples=None,
)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)
print(f'✅ Random Forest Regressor  | R²={r2_score(y_test, y_pred_rf):.4f}')
print(f'   OOB R² Score            : {rf_reg.oob_score_:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 8: GRADIENT BOOSTING REGRESSOR
# Parameters:
#   loss             : str   — 'squared_error','absolute_error','huber','quantile'
#   learning_rate    : float — shrinks tree contribution             (default 0.1)
#   n_estimators     : int   — number of boosting stages             (default 100)
#   subsample        : float — fraction of samples for base learner  (default 1.0)
#   criterion        : str   — 'friedman_mse','squared_error'        (default 'friedman_mse')
#   min_samples_split: int                                           (default 2)
#   min_samples_leaf : int                                           (default 1)
#   min_weight_fraction_leaf: float                                  (default 0.0)
#   max_depth        : int   — max depth of each tree                (default 3)
#   min_impurity_decrease: float                                     (default 0.0)
#   init             : estimator — initial estimator                 (default None)
#   random_state     : int                                           (default None)
#   max_features     : str/int/float                                 (default None)
#   alpha            : float — for 'huber' and 'quantile' loss       (default 0.9)
#   max_leaf_nodes   : int                                           (default None)
#   warm_start       : bool                                          (default False)
#   validation_fraction: float — fraction for early stopping         (default 0.1)
#   n_iter_no_change : int   — early stopping rounds                 (default None)
#   tol              : float — tolerance for early stopping          (default 1e-4)
#   ccp_alpha        : float                                         (default 0.0)

gbr = GradientBoostingRegressor(
    loss='squared_error',
    learning_rate=0.1,
    n_estimators=200,
    subsample=1.0,
    criterion='friedman_mse',
    min_samples_split=5,
    min_samples_leaf=3,
    min_weight_fraction_leaf=0.0,
    max_depth=4,
    min_impurity_decrease=0.0,
    init=None,
    random_state=42,
    max_features=None,
    alpha=0.9,
    max_leaf_nodes=None,
    warm_start=False,
    validation_fraction=0.1,
    n_iter_no_change=None,
    tol=1e-4,
    ccp_alpha=0.0,
)
gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_test)
print(f'✅ Gradient Boosting Reg.   | R²={r2_score(y_test, y_pred_gbr):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 9: SUPPORT VECTOR REGRESSOR (SVR)
# Parameters:
#   kernel   : str   — 'linear','poly','rbf','sigmoid','precomputed'  (default 'rbf')
#   degree   : int   — degree for 'poly' kernel                       (default 3)
#   gamma    : str   — 'scale','auto' or float                        (default 'scale')
#   coef0    : float — independent term in kernel func               (default 0.0)
#   tol      : float — tolerance for stopping                         (default 1e-3)
#   C        : float — regularisation (smaller = more regularised)    (default 1.0)
#   epsilon  : float — epsilon-tube width (insensitive zone)          (default 0.1)
#   shrinking: bool  — use shrinking heuristic                        (default True)
#   cache_size: float— kernel cache size in MB                        (default 200)
#   verbose  : bool                                                   (default False)
#   max_iter : int   — hard iteration limit (-1 = no limit)           (default -1)

svr = SVR(
    kernel='rbf',
    degree=3,
    gamma='scale',
    coef0=0.0,
    tol=1e-3,
    C=100.0,
    epsilon=0.1,
    shrinking=True,
    cache_size=200,
    verbose=False,
    max_iter=-1,
)
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)
print(f'✅ SVR (RBF kernel)         | R²={r2_score(y_test, y_pred_svr):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 10: K-NEAREST NEIGHBOURS REGRESSOR (KNR)
# Parameters:
#   n_neighbors   : int    — number of neighbours K                   (default 5)
#   weights       : str    — 'uniform' or 'distance'                  (default 'uniform')
#   algorithm     : str    — 'ball_tree','kd_tree','brute','auto'     (default 'auto')
#   leaf_size     : int    — leaf size for BallTree/KDTree             (default 30)
#   p             : int    — power param (1=Manhattan, 2=Euclidean)   (default 2)
#   metric        : str    — distance metric                          (default 'minkowski')
#   metric_params : dict   — additional metric params                 (default None)
#   n_jobs        : int    — parallel jobs                            (default None)

knr = KNeighborsRegressor(
    n_neighbors=7,
    weights='distance',
    algorithm='auto',
    leaf_size=30,
    p=2,
    metric='minkowski',
    metric_params=None,
    n_jobs=-1,
)
knr.fit(X_train, y_train)
y_pred_knr = knr.predict(X_test)
print(f'✅ KNN Regressor (K=7)      | R²={r2_score(y_test, y_pred_knr):.4f}')

## STEP 6 — Evaluate All Regression Models

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Helper: Adjusted R²
def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

n_test = len(y_test)
n_feat = X_test.shape[1]

MODELS_INFO = [
    ('Linear Regression',       y_pred_lr,    lr),
    ('Ridge Regression',        y_pred_ridge, ridge),
    ('Lasso Regression',        y_pred_lasso, lasso),
    ('ElasticNet Regression',   y_pred_enet,  enet),
    ('Polynomial Regression',   y_pred_poly,  poly_reg),
    ('Decision Tree Regressor', y_pred_dt,    dt_reg),
    ('Random Forest Regressor', y_pred_rf,    rf_reg),
    ('Gradient Boosting Reg.',  y_pred_gbr,   gbr),
    ('SVR (RBF)',               y_pred_svr,   svr),
    ('KNN Regressor',           y_pred_knr,   knr),
]

results = []
header = f"{'Model':<26} {'MAE':>10} {'RMSE':>10} {'R²':>8} {'Adj R²':>9} {'MAPE%':>8}"
print(header)
print('─' * 75)

for name, preds, _ in MODELS_INFO:
    mae  = mean_absolute_error(y_test, preds)
    mse  = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, preds)
    adj  = adjusted_r2(r2, n_test, n_feat)
    mape = mean_absolute_percentage_error(y_test, preds) * 100
    results.append({
        'Model': name, 'MAE': mae, 'MSE': mse,
        'RMSE': rmse, 'R2': r2, 'Adj_R2': adj, 'MAPE': mape
    })
    print(f"{name:<26} ${mae:>9,.0f} ${rmse:>9,.0f} {r2:>8.4f} {adj:>9.4f} {mape:>7.2f}%")

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print(f'\n🏆 Best Model: {results_df.iloc[0]["Model"]}  (R²={results_df.iloc[0]["R2"]:.4f})')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVAL Plot 1 — R² and RMSE Comparison Bar Charts
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('📊 Regression Model Performance Comparison',
             fontsize=15, fontweight='bold')

colors = [PALETTE[i % len(PALETTE)] for i in range(len(results_df))]

# R² bar chart (horizontal)
bars1 = axes[0].barh(results_df['Model'], results_df['R2'],
                      color=colors, edgecolor='white', height=0.65)
axes[0].set_xlabel('R² Score', fontsize=11)
axes[0].set_title('R² Score (higher is better)', fontweight='bold')
axes[0].set_xlim(0, 1.05)
axes[0].axvline(0.8, color='gray', linestyle=':', lw=1.2, label='0.8 threshold')
axes[0].legend(fontsize=9)
for bar in bars1:
    w = bar.get_width()
    axes[0].text(w + 0.008, bar.get_y() + bar.get_height()/2,
                 f'{w:.4f}', va='center', fontsize=8, fontweight='bold')

# RMSE bar chart (horizontal)
bars2 = axes[1].barh(results_df['Model'], results_df['RMSE'],
                      color=colors, edgecolor='white', height=0.65)
axes[1].set_xlabel('RMSE (USD)', fontsize=11)
axes[1].set_title('RMSE — Root Mean Squared Error (lower is better)',
                  fontweight='bold')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(
    lambda x, _: f'${x/1000:.0f}K'))
for bar in bars2:
    w = bar.get_width()
    axes[1].text(w + 200, bar.get_y() + bar.get_height()/2,
                 f'${w:,.0f}', va='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('eval_1_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Eval Plot 1 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVAL Plot 2 — Multi-metric grouped bar chart
fig, ax = plt.subplots(figsize=(18, 6))
ax.set_title('📊 All Regression Metrics per Model',
             fontsize=15, fontweight='bold')

x = np.arange(len(results_df))
# Normalise MAE and RMSE to [0,1] for display alongside R²
r2_vals   = results_df['R2'].values
adjr2_vals= results_df['Adj_R2'].values
mape_vals = results_df['MAPE'].values / 100   # convert to fraction

w = 0.2
ax.bar(x - w,   r2_vals,    w, label='R²',             color=PALETTE[0], alpha=0.9)
ax.bar(x,       adjr2_vals, w, label='Adjusted R²',    color=PALETTE[1], alpha=0.9)
ax.bar(x + w,   1-mape_vals,w, label='1−MAPE (proxy)', color=PALETTE[2], alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score (0–1 scale)', fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(0.8, color='gray', linestyle=':', lw=1.2)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('eval_2_metrics_grouped.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Eval Plot 2 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVAL Plot 3 — Actual vs Predicted scatter for all 10 models
fig, axes = plt.subplots(2, 5, figsize=(26, 10))
fig.suptitle('📈 Actual vs Predicted Salary — All Models',
             fontsize=16, fontweight='bold', y=1.01)

for ax, (name, preds, _) in zip(axes.flatten(), MODELS_INFO):
    r2   = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    ax.scatter(y_test, preds, alpha=0.45, s=18, color='#065A82', edgecolors='none')
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual ($)', fontsize=9)
    ax.set_ylabel('Predicted ($)', fontsize=9)
    ax.set_title(f'{name}\nR²={r2:.4f}  RMSE=${rmse:,.0f}', fontsize=9, fontweight='bold')
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('eval_3_actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Eval Plot 3 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVAL Plot 4 — Residual plots for all 10 models
fig, axes = plt.subplots(2, 5, figsize=(26, 10))
fig.suptitle('🔍 Residual Analysis — All Models',
             fontsize=16, fontweight='bold', y=1.01)

for ax, (name, preds, _) in zip(axes.flatten(), MODELS_INFO):
    residuals = y_test - preds
    ax.scatter(preds, residuals, alpha=0.45, s=18, color='#2C5F2D', edgecolors='none')
    ax.axhline(0, color='crimson', linestyle='--', lw=1.5)
    ax.set_xlabel('Predicted Salary', fontsize=9)
    ax.set_ylabel('Residuals (Actual − Predicted)', fontsize=9)
    ax.set_title(f'{name}', fontsize=9, fontweight='bold')
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('eval_4_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Eval Plot 4 saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVAL Plot 5 — Residual Distribution (histogram + KDE) for all models
fig, axes = plt.subplots(2, 5, figsize=(26, 10))
fig.suptitle('📉 Residual Distribution — All Models',
             fontsize=16, fontweight='bold', y=1.01)

for ax, (name, preds, _), color in zip(axes.flatten(), MODELS_INFO, PALETTE*2):
    residuals = y_test - preds
    sns.histplot(residuals, bins=25, kde=True, color=color,
                 ax=ax, edgecolor='white', alpha=0.8, linewidth=0.5)
    ax.axvline(0, color='crimson', linestyle='--', lw=1.5)
    ax.axvline(residuals.mean(), color='navy', linestyle=':', lw=1.5,
               label=f'Mean: ${residuals.mean():,.0f}')
    ax.set_title(f'{name}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Residual (USD)', fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('eval_5_residual_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Eval Plot 5 saved!')

## STEP 7 — Feature Importance & Cross-Validation

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Feature Importance from Random Forest and Gradient Boosting
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('🔍 Feature Importance — Tree-Based Regressors',
             fontsize=14, fontweight='bold')

for ax, model, title in [
    (axes[0], rf_reg,  'Random Forest Regressor'),
    (axes[1], gbr,     'Gradient Boosting Regressor'),
]:
    fi = pd.DataFrame({'Feature': FEAT_LABELS,
                       'Importance': model.feature_importances_})
    fi = fi.sort_values('Importance', ascending=True)
    colors_fi = [PALETTE[i % len(PALETTE)] for i in range(len(fi))]
    bars = ax.barh(fi['Feature'], fi['Importance'],
                   color=colors_fi, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance Score')
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.001, bar.get_y() + bar.get_height()/2,
                f'{w:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eval_6_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Feature importance plot saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 5-Fold Cross-Validation (R² scoring) for all models
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
print(f"{'Model':<26} {'CV Mean R²':>12} {'CV Std':>10} {'95% CI':>20}")
print('─' * 72)

for name, _, model in MODELS_INFO:
    scores = cross_val_score(model, X_scaled, y,
                             cv=kf, scoring='r2', n_jobs=-1)
    ci_lo = scores.mean() - 1.96 * scores.std()
    ci_hi = scores.mean() + 1.96 * scores.std()
    cv_results.append({'Model': name, 'Mean': scores.mean(),
                       'Std': scores.std(), 'Scores': scores})
    print(f"{name:<26} {scores.mean():>12.4f} {scores.std():>10.4f} "
          f"[{ci_lo:.4f}, {ci_hi:.4f}]")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CV Plot — Mean R² with error bars
cv_df = pd.DataFrame(cv_results).sort_values('Mean', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('📊 5-Fold Cross-Validation Results',
             fontsize=14, fontweight='bold')

# Bar chart with error bars
colors_cv = [PALETTE[i % len(PALETTE)] for i in range(len(cv_df))]
axes[0].barh(cv_df['Model'], cv_df['Mean'],
             xerr=cv_df['Std'] * 1.96,
             color=colors_cv, edgecolor='white', height=0.6,
             error_kw=dict(ecolor='black', lw=1.5, capsize=5))
axes[0].set_xlabel('Mean R² Score (±95% CI)', fontsize=11)
axes[0].set_title('CV R² per Model (sorted)', fontweight='bold')
axes[0].axvline(0.8, color='gray', linestyle=':', lw=1.2)

# Box plots of 5-fold scores
all_scores = [r['Scores'] for _, r in cv_df.iterrows()]
bp = axes[1].boxplot(all_scores, vert=False, patch_artist=True,
                     labels=cv_df['Model'].tolist(),
                     flierprops=dict(marker='o', markersize=4, alpha=0.5))
for patch, color in zip(bp['boxes'], colors_cv):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].axvline(0.8, color='gray', linestyle=':', lw=1.2)
axes[1].set_xlabel('R² Score across 5 folds', fontsize=11)
axes[1].set_title('Score Distribution per Model', fontweight='bold')

plt.tight_layout()
plt.savefig('eval_7_cross_validation.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ CV plot saved!')

## STEP 8 — Hyperparameter Tuning with GridSearchCV

In [ ]:
# ══════════════════════════════════════════════════════════════════
# GridSearchCV for Ridge, Lasso, ElasticNet, GradientBoosting
# (Decision Tree and Random Forest skipped here to save time;
#  add their param grids below if needed)

grid_specs = [
    ('Ridge',        Ridge(random_state=42),
     {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
      'solver': ['auto', 'svd', 'lsqr']}),

    ('Lasso',        Lasso(random_state=42, max_iter=5000),
     {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
      'selection': ['cyclic', 'random']}),

    ('ElasticNet',   ElasticNet(random_state=42, max_iter=5000),
     {'alpha': [0.01, 0.1, 1.0],
      'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]}),

    ('GradBoost',    GradientBoostingRegressor(random_state=42),
     {'n_estimators': [100, 200],
      'max_depth'   : [3, 4, 5],
      'learning_rate': [0.05, 0.1, 0.2]}),
]

best_grid_models = {}
grid_results = []

for name, estimator, param_grid in grid_specs:
    gs = GridSearchCV(estimator, param_grid, cv=5,
                      scoring='r2', n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)
    best_grid_models[name] = gs.best_estimator_
    y_gs_pred = gs.predict(X_test)
    r2_gs = r2_score(y_test, y_gs_pred)
    rmse_gs = np.sqrt(mean_squared_error(y_test, y_gs_pred))
    grid_results.append({'Model': name, 'Best Params': gs.best_params_,
                         'CV R²': gs.best_score_, 'Test R²': r2_gs,
                         'RMSE': rmse_gs})
    print(f'✅ {name:<12} | Best CV R²={gs.best_score_:.4f} | Test R²={r2_gs:.4f}')
    print(f'   Best params: {gs.best_params_}')

print('\n✅ GridSearchCV complete!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# GridSearchCV Plot — CV vs Test R² comparison
gs_df = pd.DataFrame(grid_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title('🔧 GridSearchCV — Best CV R² vs Test R²',
             fontsize=13, fontweight='bold')

x = np.arange(len(gs_df))
w = 0.3
ax.bar(x - w/2, gs_df['CV R²'],  w, label='Best CV R²',  color=PALETTE[1], alpha=0.9)
ax.bar(x + w/2, gs_df['Test R²'],w, label='Test R²',     color=PALETTE[0], alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(gs_df['Model'], fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('R² Score')
ax.axhline(0.8, color='gray', linestyle=':', lw=1.2)
ax.legend(fontsize=10)

for i, row in gs_df.iterrows():
    ax.text(i - w/2, row['CV R²'] + 0.01,
            f"{row['CV R²']:.3f}", ha='center', fontsize=9)
    ax.text(i + w/2, row['Test R²'] + 0.01,
            f"{row['Test R²']:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('eval_8_gridsearch.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ GridSearchCV plot saved!')

## STEP 9 — Ridge/Lasso Regularisation Path (Alpha Analysis)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Plot how coefficients change as alpha increases (Ridge vs Lasso)
alphas = np.logspace(-3, 4, 60)

ridge_coefs, lasso_coefs = [], []
for a in alphas:
    r = Ridge(alpha=a, fit_intercept=True).fit(X_train, y_train)
    l = Lasso(alpha=a, max_iter=10000).fit(X_train, y_train)
    ridge_coefs.append(r.coef_)
    lasso_coefs.append(l.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('📉 Regularisation Path — Ridge vs Lasso',
             fontsize=14, fontweight='bold')

for i, feat in enumerate(FEAT_LABELS):
    axes[0].semilogx(alphas, ridge_coefs[:, i], label=feat, lw=1.8)
    axes[1].semilogx(alphas, lasso_coefs[:, i], label=feat, lw=1.8)

for ax, title in [(axes[0], 'Ridge (L2) — Coefficients vs α'),
                   (axes[1], 'Lasso (L1) — Coefficients vs α')]:
    ax.set_xlabel('Regularisation Strength α (log scale)')
    ax.set_ylabel('Coefficient Value')
    ax.set_title(title, fontweight='bold')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('eval_9_regularisation_path.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Regularisation path plot saved!')

## STEP 10 — Salary Prediction Interface

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Pick best model by Test R²
best_idx   = results_df.iloc[0]['Model']
best_model = next(m for n, _, m in MODELS_INFO if n == best_idx)

print(f'✅ Best Model for Prediction : {best_idx}')
print(f'   Education options : {list(le_edu.classes_)}')
print(f'   Location options  : {list(le_loc.classes_)}')
print(f'   Job Title options : {list(le_job.classes_)}')
print(f'   Gender options    : {list(le_gen.classes_)}')


def predict_salary(education: str, experience: int, location: str,
                   job_title: str, age: int, gender: str,
                   model=best_model) -> dict:
    """
    Predict exact salary using the best regression model.

    Parameters
    ----------
    education   : str  — e.g. 'Bachelor', 'Master', 'PhD', 'High School'
    experience  : int  — years of experience
    location    : str  — 'Urban', 'Suburban', 'Rural'
    job_title   : str  — 'Analyst', 'Engineer', 'Manager', 'Director'
    age         : int  — age in years
    gender      : str  — 'Male', 'Female'
    model       : regressor — any fitted regression model

    Returns
    -------
    dict with predicted salary, model name, and input details
    """
    safe_transform = lambda le, val, col: (
        le.transform([val])[0]
        if val in le.classes_
        else (_ for _ in ()).throw(
            ValueError(f'Unknown {col}: {val}. Options: {list(le.classes_)}')
        )
    )
    try:
        edu_enc = le_edu.transform([education])[0]
        loc_enc = le_loc.transform([location])[0]
        job_enc = le_job.transform([job_title])[0]
        gen_enc = le_gen.transform([gender])[0]
    except ValueError as e:
        print(f'❌ {e}'); return None

    feat = np.array([[edu_enc, experience, loc_enc, job_enc, age, gen_enc]])
    feat_scaled = scaler.transform(feat)
    salary = model.predict(feat_scaled)[0]

    print('\n💼 SALARY PREDICTION RESULT')
    print('=' * 52)
    print(f'  Education   : {education}')
    print(f'  Experience  : {experience} years')
    print(f'  Location    : {location}')
    print(f'  Job Title   : {job_title}')
    print(f'  Age         : {age}')
    print(f'  Gender      : {gender}')
    print('─' * 52)
    print(f'  💰 Predicted Salary : ${salary:,.2f}')
    print(f'  🤖 Model Used       : {best_idx}')
    print('=' * 52)

    return {'predicted_salary': round(salary, 2), 'model': best_idx}


# ── Sample Predictions ─────────────────────────────────────────
predict_salary('PhD',      15, 'Urban',    'Director', 42, 'Male')
predict_salary('Bachelor',  5, 'Rural',    'Analyst',  28, 'Female')
predict_salary('Master',   10, 'Suburban', 'Engineer', 35, 'Male')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Compare all models on the three sample predictions
samples = [
    ('PhD',      15, 'Urban',    'Director', 42, 'Male'),
    ('Bachelor',  5, 'Rural',    'Analyst',  28, 'Female'),
    ('Master',   10, 'Suburban', 'Engineer', 35, 'Male'),
]

sample_labels = [
    'PhD / 15yr / Director',
    'Bachelor / 5yr / Analyst',
    'Master / 10yr / Engineer',
]

comparison = {}
for name, _, model in MODELS_INFO:
    preds_s = []
    for edu, exp, loc, job, age, gen in samples:
        enc_edu = le_edu.transform([edu])[0]
        enc_loc = le_loc.transform([loc])[0]
        enc_job = le_job.transform([job])[0]
        enc_gen = le_gen.transform([gen])[0]
        feat = scaler.transform([[enc_edu, exp, enc_loc, enc_job, age, enc_gen]])
        preds_s.append(model.predict(feat)[0])
    comparison[name] = preds_s

comp_df = pd.DataFrame(comparison, index=sample_labels).T

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_title('💼 Salary Predictions — All Models vs 3 Sample Profiles',
             fontsize=13, fontweight='bold')

x = np.arange(len(MODELS_INFO))
w = 0.25
for i, (label, color) in enumerate(zip(sample_labels, PALETTE)):
    vals = [comparison[n][i] for n, _, _ in MODELS_INFO]
    ax.bar(x + i*w, vals, w, label=label, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + w)
ax.set_xticklabels([n for n, _, _ in MODELS_INFO], rotation=28, ha='right', fontsize=9)
ax.set_ylabel('Predicted Salary (USD)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('eval_10_sample_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sample predictions comparison plot saved!')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Interactive CLI Predictor (uncomment to use)
def get_user_prediction():
    print('--- 💼 AI Salary Predictor (Regression) ---')
    edu  = input(f'Education {list(le_edu.classes_)}: ').strip()
    exp  = int(input('Years of Experience: ').strip())
    loc  = input(f'Location {list(le_loc.classes_)}: ').strip()
    job  = input(f'Job Title {list(le_job.classes_)}: ').strip()
    age  = int(input('Age: ').strip())
    gen  = input(f'Gender {list(le_gen.classes_)}: ').strip()
    return predict_salary(edu, exp, loc, job, age, gen)

# Uncomment to run:
# get_user_prediction()

## STEP 11 — Push to GitHub

In [ ]:
# ══════════════════════════════════════════════════════════════════
"""
YOUR_NAME   = 'Your Name'
YOUR_EMAIL  = 'your@email.com'
YOUR_GITHUB = 'your-github-username'
REPO_NAME   = 'ml-salary-prediction-regression'
YOUR_TOKEN  = 'ghp_xxxxxxxxxxxxxxxxxx'

!git config --global user.name  "{YOUR_NAME}"
!git config --global user.email "{YOUR_EMAIL}"
!git init
!git add .
!git commit -m 'Salary Prediction — Regression Only — Group 9'
!git branch -M main
!git remote add origin https://{YOUR_TOKEN}@github.com/{YOUR_GITHUB}/{REPO_NAME}.git
!git push -u origin main
"""

## SUMMARY — Regression Models & Parameters Used

```
+=========================================================================+
|          SALARY PREDICTION — REGRESSION ONLY — PROJECT SUMMARY         |
+=========================================================================+
|  Model                    | Key Parameters                | Syllabus   |
| ----------------------------------------------------------------------- |
|  Linear Regression        | fit_intercept,n_jobs,positive | Unit I     |
|  Ridge Regression         | alpha,solver,tol,max_iter     | Unit I     |
|  Lasso Regression         | alpha,selection,warm_start    | Unit I     |
|  ElasticNet               | alpha,l1_ratio,selection      | Unit I     |
|  Polynomial Regression    | degree,interaction_only       | Unit I     |
|  Decision Tree Regressor  | max_depth,criterion,splitter  | Unit II    |
|  Random Forest Regressor  | n_estimators,oob_score        | Unit II    |
|  Gradient Boosting Reg.   | loss,learning_rate,subsample  | Unit II    |
|  SVR                      | kernel,C,epsilon,gamma        | Unit II    |
|  KNN Regressor            | n_neighbors,weights,metric    | Unit III   |
+=========================================================================+
|  Evaluation Metrics: MAE, MSE, RMSE, R², Adjusted R², MAPE            |
|  Tuning: GridSearchCV (5-fold) for Ridge, Lasso, ElasticNet, GBR      |
|  CV:     KFold 5-fold cross-validation for all models                  |
|  Plots:  Distributions · Violin · Correlation · Actual-vs-Predicted   |
|          Residuals · Residual Dist. · Feature Importance · CV · Path   |
+=========================================================================+
```